# Experiment: Seqrec Constrained Results Analysis (Mean Only)

目标：
- 只读取并统计 `mean_results`
- 不再处理 `min_results` 和 `max_results`
- `model_dir + checkpoint` 视为不同模型（独立统计）


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd

try:
    import seaborn as sns

    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

DEFAULT_RESULTS_ROOT = Path(
    "/mnt/dolphinfs/hdd_pool/docker/user/hadoop-hmart-poistar/fanghaotian/GRec/results/test/seqrec-constrained"
)
RESULTS_ROOT = DEFAULT_RESULTS_ROOT

# 可选：只看某个 index_tag。None 表示不过滤。
INDEX_TAG_FILTER: str | None = None

print(f"Using RESULTS_ROOT: {RESULTS_ROOT}")
print(f"Seaborn available: {HAS_SEABORN}")
print(f"INDEX_TAG_FILTER: {INDEX_TAG_FILTER}")

## 1) 扫描 results.json

In [ ]:
result_files = sorted(RESULTS_ROOT.rglob("results.json"))
print(f"Found {len(result_files)} results.json files")
for path in result_files[:8]:
    print(" -", path)
if len(result_files) > 8:
    print(" ...")

if not result_files:
    raise FileNotFoundError(f"No results.json found under: {RESULTS_ROOT}")

In [ ]:
def parse_results_file(path: Path, root: Path) -> dict[str, Any]:
    rel = path.relative_to(root)
    parts = rel.parts
    payload = json.loads(path.read_text(encoding="utf-8"))

    index_tag = parts[0] if len(parts) >= 1 else ""
    model_dir = parts[1] if len(parts) >= 2 else ""
    checkpoint = parts[2] if len(parts) >= 3 else ""
    model_key = f"{index_tag} | {model_dir} | {checkpoint}"

    return {
        "path": str(path),
        "index_tag": index_tag,
        "model_dir": model_dir,
        "checkpoint": checkpoint,
        "model_key": model_key,
        "eval_split": payload.get("eval_split"),
        "rollout_cached": payload.get("rollout_cached"),
        "test_prompt_ids": payload.get("test_prompt_ids"),
        "mean_results": payload.get("mean_results", {}) or {},
    }


records = [parse_results_file(path, RESULTS_ROOT) for path in result_files]
runs_df = pd.DataFrame(records)

if INDEX_TAG_FILTER:
    runs_df = runs_df[runs_df["index_tag"] == INDEX_TAG_FILTER].copy()

runs_df["file_mtime"] = runs_df["path"].map(lambda p: Path(p).stat().st_mtime)
runs_df = runs_df.sort_values(
    ["index_tag", "model_dir", "checkpoint", "file_mtime"]
)

print(f"Loaded runs after filter: {len(runs_df)}")
runs_df[
    ["index_tag", "model_dir", "checkpoint", "rollout_cached", "path"]
].head(12)

## 2) 只展开 mean_results

In [ ]:
metric_rows = []

for _, row in runs_df.iterrows():
    mean_results = (
        row["mean_results"] if isinstance(row["mean_results"], dict) else {}
    )
    for metric, value in mean_results.items():
        try:
            value = float(value)
        except (TypeError, ValueError):
            continue

        metric_rows.append(
            {
                "index_tag": row["index_tag"],
                "model_dir": row["model_dir"],
                "checkpoint": row["checkpoint"],
                "model_key": row["model_key"],
                "metric": metric,
                "mean_value": value,
                "path": row["path"],
                "file_mtime": row["file_mtime"],
            }
        )

metrics_df = pd.DataFrame(metric_rows)
if metrics_df.empty:
    raise ValueError("No mean metrics extracted from results.json")

print(f"Extracted mean metric rows: {len(metrics_df)}")
metrics_df.head(20)

In [ ]:
# 每个 model_key(=index_tag|model_dir|checkpoint) 独立保留，不做 min/max
model_metric_df = metrics_df[
    [
        "model_key",
        "index_tag",
        "model_dir",
        "checkpoint",
        "metric",
        "mean_value",
        "path",
    ]
].copy()
model_metric_df = model_metric_df.sort_values(["model_key", "metric"])

pivot_mean_df = model_metric_df.pivot_table(
    index="model_key",
    columns="metric",
    values="mean_value",
    aggfunc="mean",
)

print(f"Unique model_key count: {pivot_mean_df.shape[0]}")
pivot_mean_df.head(20)

## 3) 画图（每个 checkpoint 作为独立模型）

In [ ]:
preferred_metrics = ["hit@1", "hit@10", "hit@50", "ndcg@10", "ndcg@50"]
available_metrics = [m for m in preferred_metrics if m in pivot_mean_df.columns]
if not available_metrics:
    available_metrics = list(pivot_mean_df.columns[:5])

bar_df = pivot_mean_df[available_metrics].copy()
ax = bar_df.plot(kind="bar", figsize=(16, 6))
ax.set_title("Mean Metrics by Model+Checkpoint")
ax.set_xlabel("model_key")
ax.set_ylabel("mean metric value")
ax.legend(title="metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
heatmap_df = pivot_mean_df.copy()
fig_w = max(10, 0.75 * len(heatmap_df.columns))
fig_h = max(5, 0.35 * len(heatmap_df.index))
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

if HAS_SEABORN:
    sns.heatmap(heatmap_df, annot=True, fmt=".4f", cmap="YlGnBu", ax=ax)
else:
    im = ax.imshow(heatmap_df.values, aspect="auto", cmap="YlGnBu")
    ax.set_xticks(range(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index)
    fig.colorbar(im, ax=ax)

ax.set_title("Mean Metrics Heatmap by Model+Checkpoint")
plt.tight_layout()
plt.show()

## 4) 导出统计结果

In [ ]:
analysis_dir = RESULTS_ROOT / "_analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

runs_out = analysis_dir / "runs_summary_mean_only.csv"
long_out = analysis_dir / "metrics_long_mean_only.csv"
pivot_out = analysis_dir / "metrics_pivot_mean_only.csv"

runs_export_df = runs_df.drop(columns=["mean_results"])
runs_export_df.to_csv(runs_out, index=False)
model_metric_df.to_csv(long_out, index=False)
pivot_mean_df.reset_index().to_csv(pivot_out, index=False)

print("Exported:")
print(" -", runs_out)
print(" -", long_out)
print(" -", pivot_out)